# Análise de Inferência Causal usando DoWhy

Este notebook carrega o conjunto de dados em `data/processed/farms.csv`, importa a definição do DAG causal do arquivo `model/causal_dag.py` e executa tarefas de inferência causal utilizando a biblioteca **DoWhy** para analisar qual cluster de dieta das fazendas oferece o melhor trade-off produtividade e emissões entéricas (buscando maior produtividade e menor emissões).

In [3]:
import os
import sys
import re
import pandas as pd
import numpy as np
import dowhy
from dowhy import CausalModel

# Garante que o diretório de execução seja a raiz do projeto para localizar os arquivos corretamente
if os.getcwd().endswith('notebooks'):
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

# Exibir todas as colunas do dataset
pd.set_option('display.max_columns', None)

/home/vscode/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Carregando os Dados

In [4]:
df = pd.read_csv('data/processed/farms.csv', sep=';')
print(f"Dimensões do dataset: {df.shape}")
df.head()

Dimensões do dataset: (50, 39)


,Property_Name,Location,Owner_Name,Information_Responsible,Predominant_Soil_Type,Lactating_Cows,Dry_Cows,Heifers,Calves,Bulls,Lactating_Cows_Weight,Dry_Cows_Weight,Heifers_Weight,Calves_Weight,Bulls_Weight,Females,Males,Cows_Discarded_Per_Year,Discarded_Cows_Destination,Productivity,Total_Milk_Production,Milk_Protein_Percentage,Milk_Fat_Percentage,Total,DMI_Check,Diesel,Gasoline,Other_Fuel,Electricity_Grid,Photovoltaic_Energy,Pasture_Area,Non_Organic_Fertilizer,Organic_Fertilizer,co2_enteric_fermentation,co2_manure_management,co2_fertilizer_emissions,co2_energy_emissions,co2_total_emissions,Dietary_Strategy_Cluster
0,BR – 1,Santa Catarina,Anonymous,Anonymous,Latossolo Vermelho-Amarelo,13.0,4.0,8.0,15.0,0.0,500.0,530.0,320.0,150.0,0.0,0.0,0.0,4.0,venda,15.465629,73384.41,3.32,4.13,88410.30,88410.30,0.0,0.0,0.0,2500.0,0.0,3.0,2304.0,0.0,51000.0,3471.45,7209.2160,582.5,62263.1660,0
1,BR – 2,Santa Catarina,Anonymous,Anonymous,Latossolo Vermelho-Amarelo,26.0,5.0,12.0,19.0,0.0,500.0,530.0,320.0,150.0,0.0,0.0,0.0,15.0,venda,17.117211,162442.33,3.19,3.96,215423.00,215423.00,2000.0,0.0,0.0,9600.0,0.0,3.5,455.0,0.0,89750.0,6116.70,1423.6950,7596.8,104887.1950,1
2,BR – 3,Santa Catarina,Anonymous,Anonymous,Latossolo Vermelho-Amarelo,30.0,6.0,20.0,27.0,0.0,500.0,530.0,320.0,150.0,0.0,0.0,0.0,12.0,venda,22.772406,249357.85,2.96,3.35,291996.35,291996.35,2000.0,0.0,0.0,12000.0,0.0,4.0,854.0,0.0,111000.0,7537.35,2672.1660,8156.0,129365.5160,1
3,BR – 4,Santa Catarina,Anonymous,Anonymous,Latossolo Vermelho-Amarelo,32.0,10.0,12.0,9.0,0.0,500.0,530.0,320.0,150.0,0.0,0.0,0.0,9.0,venda,16.757430,195726.78,3.48,4.20,138356.90,138356.90,600.0,0.0,0.0,9600.0,0.0,4.0,696.0,0.0,106000.0,7598.45,2177.7840,3844.8,119621.0340,2
4,BR – 5,Santa Catarina,Anonymous,Anonymous,Latossolo Vermelho-Amarelo,30.0,4.0,13.0,20.0,0.0,500.0,530.0,320.0,150.0,0.0,0.0,0.0,5.0,venda,19.967890,218648.40,3.31,3.92,290445.10,290445.10,0.0,0.0,0.0,12000.0,0.0,3.0,433.5,0.0,99750.0,6743.20,1356.4215,2796.0,110645.6215,2


# 2. Importando o grafo causal de model/causal_dag.py

In [5]:
from model.causal_dag import causal_graph

## 3. Inicializando os Modelos Causais no DoWhy

Instanciamos 2 `CausalModel`'s:
Em ambos usamos
- **Dados (data):** Nossos dados importados
- **Tratamento (Treatment):** `Dietary_Strategy_Cluster` (perfil de estratégia dietética)
- **Gráfico Causal (Graph):** nosso grafo causal importado acima

Criamos dois modelos, pois cada um considera uma variável-alvo como saída (Y), portanto
- **Desfecho (Outcome):** `co2_enteric_fermentation` (emissões relacionadas à fermentação entérica) ou `Productivity` (produtividade dos animais medidas em L por animal)

As variáveis presentes no DAG mas ausentes no arquivo CSV (`Cattle_Breed`, `Production_System`) serão automaticamente tratadas como variáveis não observadas (confounders latentes) pelo DoWhy.

In [15]:
# Troca o tipo de variável de cluster para não ser interpretada como numérica contínua
df['Dietary_Strategy_Cluster'] = df['Dietary_Strategy_Cluster'].astype(str)

model_emissions = CausalModel(
    data=df,
    treatment='Dietary_Strategy_Cluster',
    outcome='co2_enteric_fermentation',
    graph=causal_graph   
)

model_prod = CausalModel(
    data=df,
    treatment='Dietary_Strategy_Cluster',
    outcome='Productivity',
    graph=causal_graph   
)


/home/vscode/.local/lib/python3.11/site-packages/dowhy/causal_model.py:581: UserWarning: 2 variables are assumed unobserved because they are not in the dataset. Configure the logging level to `logging.WARNING` or higher for additional details.
  warnings.warn(


# 4. Estimando efeitos com perguntas contra-factuais

Queremos simular com base em causalidade e dados uma "intervenção universal" hipotética, isto é, para cada cluster de dieta (0, 1, 2, 3), vamos perguntar ao modelo: "Quais resultados de produtividade e emissões obteríamos se todas as fazendas do dataset adotassem essa dieta, mantendo todos seus outros dados?". Com isso, nosso objetivo é extrair os resultados causais: O DoWhy calculará o valor esperado causal para a Produtividade $E[\text{Productivity} \vert{} do(\text{Cluster}=k)]$ e para as emissões $E[\text{co2\_enteric\_fermentation} \vert{} do(\text{Cluster}=k)]$ para que calculemos o Trade-Off.

In [ ]:
# Para as perguntas que queremos, usaremos a API GCM do DoWhy
import networkx as nx
import dowhy.gcm as gcm

# A API GCM precisa do grafo no formato do networkx
causal_model = gcm.InvertibleStructuralCausalModel(nx.DiGraph(nx.nx_pydot.read_dot('model/causal_dag_datarepresented.dot')))

In [21]:
gcm.auto.assign_causal_mechanisms(causal_model, df)
gcm.fit(causal_model, df)

results = {}

for cluster in sorted(df['Dietary_Strategy_Cluster'].unique()):
    samples = gcm.interventional_samples(
        causal_model,
        interventions={'Dietary_Strategy_Cluster': lambda x: cluster},
        num_samples_to_draw=1000
    )

    prod_absolute_mean = samples['Productivity'].mean()
    emissions_absolute_mean = samples['co2_enteric_fermentation'].mean()

    ratio = prod_absolute_mean / emissions_absolute_mean if emissions_absolute_mean != 0 else 0

    results[cluster] = {
        'Productivity': prod_absolute_mean,
        'Emissions': emissions_absolute_mean,
        'Ratio': ratio
    }

print(results)

KeyError: "None of [Index(['Cattle_Breed'], dtype='str')] are in the [columns]"